# Foundations NLP Block Course Assignment

Train and evaluate character-level GPT model on small Shakespeare dataset using nanoGPT

Student Names: Alexis Liston, Isabel Sittner
Course: Foundations of NLP
WiSe 25/26

## Notebook Outline
1. Environment setup and verification
2. Baseline trianing
3. Experiment with hyperparameters
4. Log parsing, plots
5. Samples and qualitative evaluation
6. Analysis

### 1. Environment Setup and Verification

In [ ]:
# verify environment setup

import sys
import torch
import numpy as np
# just a comment
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")

Python version: 3.11.15
PyTorch version: 2.11.0+cu130
NumPy version: 2.4.3
CUDA available: True
CUDA device: NVIDIA RTX A1000 6GB Laptop GPU
CUDA version (PyTorch): 13.0


In [2]:
# verify correct working directory and folders

import os
print(f"Current working directory: {os.getcwd()}")
print(f"train.py exists: {os.path.exists('train.py')}")
print(f"data/shakespeare_char/train.bin exists: {os.path.exists('data/shakespeare_char/train.bin')}")
print(f"logs/ exists: {os.path.exists('logs')}")
print(f"config/ exists: {os.path.exists('config')}")

Current working directory: c:\Users\lexil\code\NLP Foundations Assignment\FoundationsNLP
train.py exists: True
data/shakespeare_char/train.bin exists: True
logs/ exists: True
config/ exists: True


Prepare the dataset before baseline training (this creates the train.bin and val.bin files, the tokenized training and validation datasets)  
Run this in the terminal:

cd data/shakespeare_char  
python prepare.py  
cd ../..

### 2. Baseline Training

baseline training as specified in assignment. I'm using an NVIDIA RTX A100 GPU, so will set device = 'cuda'

In [6]:
%%writefile config/train_shakespeare_char_baseline.py
# baseline config file specified in assignment

out_dir = 'out-shakespeare-baseline'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False
wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'baseline'
dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100
weight_decay = 1e-1
device = 'cuda' # using 'cuda' with GPU
compile = False # compiler has issues on Windows; leaving False

Overwriting config/train_shakespeare_char_baseline.py


In [ ]:
# run the baseline training
# pipe output to the notebook display and a log file (baseline.log) and track training runtime

import subprocess
import time

log_path = "logs/baseline.log"
cmd = ["python", "train.py", "config/train_shakespeare_char_baseline.py"]

start_time = time.time()

with open(log_path, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")          # live display in the notebook
        log_file.write(line)         # simultaneously write to log file
        log_file.flush()             # flush so the file updates in real time
    process.wait()

elapsed = time.time() - start_time
minutes, seconds = divmod(elapsed, 60)

summary = (
    f"\n--- Training finished with exit code {process.returncode} ---\n"
    f"Wall-clock time: {int(minutes)} min {seconds:.1f} sec ({elapsed:.1f} s total)\n"
)
print(summary)

# Append the timing to the log file so it's preserved alongside the training output
with open(log_path, "a", encoding="utf-8") as log_file:
    log_file.write(summary)

In [7]:
# Sample generation from the trained model

import subprocess

sample_output_path = "logs/baseline_samples.txt"
cmd = [
    "python", "sample.py",
    "--out_dir=out-shakespeare-baseline",
    "--device=cuda",
    "--num_samples=3",
    "--max_new_tokens=500",
]

with open(sample_output_path, "w", encoding="utf-8") as sample_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sample_file.write(line)
        sample_file.flush()
    process.wait()

print(f"\n--- Sampling finished with exit code {process.returncode} ---")
print(f"Samples saved to: {sample_output_path}")

Overriding: out_dir = out-shakespeare-baseline
Overriding: device = cuda
Overriding: num_samples = 3
Overriding: max_new_tokens = 500
number of parameters: 10.65M
Loading meta from data\shakespeare_char\meta.pkl...


ANGELO:
And cowards it be strawn'd; and you will grave you
gates your happiness, that by the shorten with me.

ISABELLA:
In gold time of it.

DUKE VINCENTIO:
Why, well, how that Hercules the former properioved me
else no more.

ANGELO:
Here's a minister.

ISABELLA:
I will answer the prisoner of the
most Capulet, prize on the dominion. I have
strange it for the king thrust for the great sheep-by allowers, and
she was a marble soldier.

ANGELO:
Had you a help?

ISABELLA:
A neither to change me to
---------------

Men pardon me, you shall have hanged to myself.

LORD FITZWATER:
The honour is so not unborn, whom comes off
Which mighty more should perjury of evils.

WARWICK:
Hence stays our father; but what makes the grounds
Of blood, against his sovereign news our household.



### 3. Hyperparameter Experiments

Starting with a resuable helper function to define funtions for running the train.py with the experiment configuration and generating text samples from the trained model with sample.py, capture run-time, live output display and saving log-files. This helps ensure consistency across experiments and avoids repeating the same boilerplate lines for each experiment.

In [7]:
# Helper function

import subprocess
import time
from pathlib import Path

def run_training(config_name):
    """
    Run a nanoGPT training experiment.
    
    Args:
        config_name: Name of the config file (without path or extension).
                     e.g. "train_shakespeare_char_lr1e2"
    
    The config file should exist at config/{config_name}.py
    Logs are saved to logs/{experiment_name}.log, where experiment_name
    is the config name with the 'train_shakespeare_char_' prefix stripped.
    """
    config_path = f"config/{config_name}.py"
    
    # give a short experiment name for the log file
    experiment_name = config_name.replace("train_shakespeare_char_", "")
    log_path = f"logs/{experiment_name}.log"
    
    print(f"{'='*60}")
    print(f"Experiment: {experiment_name}")
    print(f"Config:     {config_path}")
    print(f"Log:        {log_path}")
    print(f"{'='*60}\n")
    
    cmd = ["python", "train.py", config_path]
    start_time = time.time()
    
    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()
        process.wait()
    
    elapsed = time.time() - start_time
    minutes, seconds = divmod(elapsed, 60)
    
    summary = (
        f"\n{'='*60}\n"
        f"Experiment '{experiment_name}' finished (exit code {process.returncode})\n"
        f"Wall-clock time: {int(minutes)} min {seconds:.1f} sec ({elapsed:.1f} s total)\n"
        f"{'='*60}\n"
    )
    print(summary)
    
    with open(log_path, "a", encoding="utf-8") as log_file:
        log_file.write(summary)
    
    return process.returncode


def run_sampling(out_dir, experiment_name, num_samples=3, max_new_tokens=500):
    """
    Generate text samples from a trained model.
    
    Args:
        out_dir:         The output directory used during training (contains ckpt.pt).
                         e.g. "out-shakespeare-lr1e2"
        experiment_name: Short name for labelling the output file.
                         e.g. "lr1e2"
        num_samples:     Number of samples to generate (default: 3).
        max_new_tokens:  Tokens per sample (default: 500).
    
    Samples are saved to logs/{experiment_name}_samples.txt
    """
    sample_path = f"logs/{experiment_name}_samples.txt"
    
    print(f"{'='*60}")
    print(f"Generating {num_samples} samples from: {out_dir}")
    print(f"Saving to: {sample_path}")
    print(f"{'='*60}\n")
    
    cmd = [
        "python", "sample.py",
        f"--out_dir={out_dir}",
        "--device=cuda",
        f"--num_samples={num_samples}",
        f"--max_new_tokens={max_new_tokens}",
    ]
    
    with open(sample_path, "w", encoding="utf-8") as sample_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            sample_file.write(line)
            sample_file.flush()
        process.wait()
    
    print(f"\n--- Sampling finished (exit code {process.returncode}) ---")
    print(f"Samples saved to: {sample_path}")
    
    return process.returncode

#### Experiment 1 - High learning rate (1e-2)

Learning rate is 10x higher than the baseline. 

In [ ]:
%%writefile config/train_shakespeare_char_lr1e2.py
# Experiment 1: High learning rate
# Change: learning_rate 1e-3 -> 1e-2 (10x increase)
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-lr1e2'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'lr1e2'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-2       # CHANGED: baseline was 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda' # change to 'cpu' for CPU
compile = False

Writing config/train_shakespeare_char_lr1e2.py


In [9]:
run_training("train_shakespeare_char_lr1e2")

Experiment: lr1e2
Config:     config/train_shakespeare_char_lr1e2.py
Log:        logs/lr1e2.log

Overriding config with config/train_shakespeare_char_lr1e2.py:
# Experiment 1: High learning rate
# Change: learning_rate 1e-3 -> 1e-2 (10x increase)
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-lr1e2'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'lr1e2'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-2       # CHANGED: baseline was 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda'
compile = False

tokens per iteration will be: 16,384
found vocab_size = 65 (inside data\shakespeare_char\meta.pkl)
Initializing a new model from scratch
number of para

0

In [10]:
run_sampling("out-shakespeare-lr1e2", "lr1e2")

Generating 3 samples from: out-shakespeare-lr1e2
Saving to: logs/lr1e2_samples.txt

Overriding: out_dir = out-shakespeare-lr1e2
Overriding: device = cuda
Overriding: num_samples = 3
Overriding: max_new_tokens = 500
number of parameters: 10.65M
Loading meta from data\shakespeare_char\meta.pkl...


KING RICHARD II:
Shall I, be made the way to take and my dagger
My arm that unquite but redeem'd, every to you,
Do not but my tongue to the conceit of dign,
We might send in the stride virtue of the next
And justice lind that the crown with a crown.

DUKE OF YORK:
He came the ground many world that more counsel,
And she's the other hands in his son
The foresh pride thrust for the great shadow pins,
And that Prince will bring upon thee discoved:
Thy brother, this courtesy tears,--

DUKE OF SURRE
---------------

Men pardon me such seeming steem so plant,
Nor now fortune's majesty; and being it now
And I, with confessions to castle. And ask you?

Second Lady:
The more of the parts to the prince;

0

#### Experiment 2 - Low Learning Rate (5e-4)

Learning rate set to 5e-4

In [11]:
%%writefile config/train_shakespeare_char_lr5e4.py
# Experiment 2: Low learning rate
# Change: learning_rate 1e-3 -> 5e-4 (5x decrease)
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-lr5e4'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'lr5e4'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 5e-4       # CHANGED: baseline was 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda' # change to 'cpu' for CPU
compile = False

Writing config/train_shakespeare_char_lr5e4.py


In [12]:
run_training("train_shakespeare_char_lr5e4")

Experiment: lr5e4
Config:     config/train_shakespeare_char_lr5e4.py
Log:        logs/lr5e4.log

Overriding config with config/train_shakespeare_char_lr5e4.py:
# Experiment 2: Low learning rate
# Change: learning_rate 1e-3 -> 5e-4 (5x decrease)
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-lr5e4'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'lr5e4'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 5e-4       # CHANGED: baseline was 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda' # change to 'cpu' for CPU
compile = False

tokens per iteration will be: 16,384
found vocab_size = 65 (inside data\shakespeare_char\meta.pkl)
Initializing a new model fro

0

In [13]:
run_sampling("out-shakespeare-lr5e4", "lr5e4")

Generating 3 samples from: out-shakespeare-lr5e4
Saving to: logs/lr5e4_samples.txt

Overriding: out_dir = out-shakespeare-lr5e4
Overriding: device = cuda
Overriding: num_samples = 3
Overriding: max_new_tokens = 500
number of parameters: 10.65M
Loading meta from data\shakespeare_char\meta.pkl...


ANGELO:
And coward, is the servant of the deed?

BOP:
Sir, take him to thee upon the traitor:
Some saw cruel than the men of those proof is conceiting,
And with it enter'd with the clumny sound;
Where he is warm'd like to early his power,
Nor she is spent years to us or do this death:
I will make all all so what evils I have done.

BENVOLIO:
A poor gentleman of a mother, I would not say:
'Tis your poor fellows in my other fair proper
And allady bear the formed of my care to kiss
The hands of th
---------------

Men part of my young party, my master and seems
Some present made well careless of the ground.

BUCKINGHAM:
My lord, good my lord, that was thou ever
A prevoket--indeed, because I meet 

0

#### Experiment 3 - Isabel

dropuot

#### Experiment 4 - Higher Model Depth

model depth changed from 6 to 8

In [14]:
%%writefile config/train_shakespeare_char_md8.py
# Experiment 4: High Model Depth
# Change: model depth n_layer 6 -> 8
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-md8'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'md8'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 8              # CHANGED: baseline was 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 5e-4       
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda' # change to 'cpu' for CPU
compile = False

Writing config/train_shakespeare_char_md8.py


In [15]:
run_training("train_shakespeare_char_md8")

Experiment: md8
Config:     config/train_shakespeare_char_md8.py
Log:        logs/md8.log

Overriding config with config/train_shakespeare_char_md8.py:
# Experiment 4: High Model Depth
# Change: model depth n_layer 6 -> 8
# All other hyperparameters match baseline

out_dir = 'out-shakespeare-md8'

eval_interval = 250
eval_iters = 200            
log_interval = 10

always_save_checkpoint = False

wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'md8'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 8              # CHANGED: baseline was 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 5e-4       
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99

warmup_iters = 100
weight_decay = 1e-1

device = 'cuda' # change to 'cpu' for CPU
compile = False

tokens per iteration will be: 16,384
found vocab_size = 65 (inside data\shakespeare_char\meta.pkl)
Initializing a new model from scratch
number

0

In [16]:
run_sampling("out-shakespeare-md8", "md8")

Generating 3 samples from: out-shakespeare-md8
Saving to: logs/md8_samples.txt

Overriding: out_dir = out-shakespeare-md8
Overriding: device = cuda
Overriding: num_samples = 3
Overriding: max_new_tokens = 500
number of parameters: 14.19M
Loading meta from data\shakespeare_char\meta.pkl...

And they bring with the blessed and take about and rude
Much and her rain after her very complain,
But when I defy the more than the moon is dead.

GLOUCESTER:
Which, son, in the Hercules that Bona longer
And makes like dispassages me now by my prince.

KING HENRY VI:
What, this we have lawful to kill at him when I it so,
Some other hath hath stood more for the kingdom than
Is no more whole wretched of the field?

KING HENRY VI:
Hark, by the royal change for his blood?

QUEEN ELIZABETH:
Why so y
---------------

Men particular you now, which you bear himself,
Some he shall ready that have seen the greater,
That have yourselves of men and my thumble
Some under when my state distressed shall be thrown.

0